# PHẦN 3 – PHÂN TÍCH 3 TẬP DỮ LIỆU

## Mục tiêu

Notebook này **chỉ tập trung vào Phần 3**: trình bày, kiểm tra, phân tích đặc điểm, kiểm tra chất lượng dữ liệu, visualization và chuẩn bị dữ liệu của 3 dataset được chọn:

1. **EMNIST ByMerge** – dữ liệu ảnh ký tự viết tay.
2. **A–Z Handwritten Alphabets** – dữ liệu ảnh chữ cái viết tay.
3. **Diabetes04** – dữ liệu dạng bảng về bệnh tiểu đường.

Notebook **không xây dựng CNN, không huấn luyện mô hình và không đánh giá mô hình**. Các nội dung đó thuộc Phần 4 và Phần 5.

Cách tổ chức notebook được xây dựng theo phong cách của các notebook tham khảo đã cung cấp: kiểm tra dữ liệu trước, EDA và visualization ở giữa, sau đó mới thực hiện các bước preprocessing cần thiết.

## 3.1. Cấu hình đường dẫn

Ba file dữ liệu được đặt trong thư mục:

`C:\DATA\`

Tên file:

- `emnist-bymerge-train.csv`
- `A_Z Handwritten Data.csv`
- `diabetes04.csv`

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DATA_DIR = Path(r"C:\DATA")

EMNIST_PATH = DATA_DIR / "emnist-bymerge-train.csv"
AZ_PATH = DATA_DIR / "A_Z Handwritten Data.csv"
DIABETES_PATH = DATA_DIR / "diabetes04.csv"

for name, path in {
    "EMNIST ByMerge": EMNIST_PATH,
    "A-Z Handwritten": AZ_PATH,
    "Diabetes04": DIABETES_PATH,
}.items():
    print(f"{name:24s}: {path} | tồn tại = {path.exists()}")

if not all(path.exists() for path in [EMNIST_PATH, AZ_PATH, DIABETES_PATH]):
    raise FileNotFoundError(
        "Thiếu ít nhất một file. Hãy kiểm tra lại thư mục C:\\DATA\\ và tên file."
    )


## 3.2. Các hàm hỗ trợ

Hai dataset ảnh rất lớn. Vì vậy notebook sử dụng hai mức đọc dữ liệu:

- **Quét cột nhãn theo từng chunk** để thống kê toàn bộ phân bố lớp mà không phải giữ toàn bộ ảnh trong RAM.
- **Đọc một mẫu hữu hạn** để visualization và kiểm tra chất lượng ảnh.

Cách này phù hợp với mục tiêu của Phần 3 vì ta vẫn phân tích được toàn bộ phân bố nhãn nhưng không biến bước EDA thành một bài toán quản lý bộ nhớ.

In [ ]:
IMAGE_SAMPLE_ROWS = 5000
IMAGE_DISPLAY_COUNT = 16
IMAGE_CHUNK_SIZE = 50000
RANDOM_SEED = 42

def detect_csv_header(path, sample_lines=2):
    """
    Phát hiện tương đối file CSV có header hay không.
    Dataset ảnh thường có dòng đầu tiên là dữ liệu số.
    """
    preview = pd.read_csv(path, header=None, nrows=sample_lines)
    first_row = preview.iloc[0]

    numeric_ratio = pd.to_numeric(first_row, errors="coerce").notna().mean()
    has_header = numeric_ratio < 0.90
    return 0 if has_header else None


def prepare_image_frame(df):
    """
    Chuẩn hóa tên cột của dataframe ảnh:
    cột 0 = label, các cột còn lại = pixel.
    """
    df = df.copy()
    df = df.rename(columns={df.columns[0]: "label"})
    pixel_columns = [c for c in df.columns if c != "label"]
    rename_map = {c: f"pixel_{i:03d}" for i, c in enumerate(pixel_columns)}
    return df.rename(columns=rename_map)


def load_image_sample(path, nrows=IMAGE_SAMPLE_ROWS):
    header = detect_csv_header(path)
    df = pd.read_csv(path, header=header, nrows=nrows)
    return prepare_image_frame(df), header


def scan_label_distribution(path, chunksize=IMAGE_CHUNK_SIZE):
    """
    Quét toàn bộ file nhưng chỉ giữ cột nhãn.
    """
    header = detect_csv_header(path)
    counts = {}

    reader = pd.read_csv(
        path,
        header=header,
        usecols=[0],
        chunksize=chunksize
    )

    for chunk in reader:
        label_col = chunk.columns[0]
        vc = chunk[label_col].value_counts()

        for label, count in vc.items():
            counts[label] = counts.get(label, 0) + int(count)

    return pd.Series(counts, dtype="int64").sort_index()


def image_arrays(df):
    labels = df["label"].to_numpy()
    pixels = df.drop(columns="label").to_numpy(dtype=np.uint8)
    return labels, pixels.reshape(-1, 28, 28)


def show_image_grid(df, title, indices=None, cmap="gray", cols=4):
    labels, images = image_arrays(df)

    if indices is None:
        rng = np.random.default_rng(RANDOM_SEED)
        n = min(IMAGE_DISPLAY_COUNT, len(df))
        indices = rng.choice(len(df), size=n, replace=False)

    indices = list(indices)
    rows = int(np.ceil(len(indices) / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(10, 2.6 * rows))
    axes = np.atleast_1d(axes).ravel()

    for ax in axes:
        ax.axis("off")

    for ax, idx in zip(axes, indices):
        ax.imshow(images[idx], cmap=cmap)
        ax.set_title(f"label = {labels[idx]}")
        ax.axis("off")

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def image_quality_report(df):
    """
    Kiểm tra sơ bộ trên mẫu:
    - min/max pixel
    - mean/std
    - tỷ lệ pixel có mực (pixel > 20)
    - số ảnh gần như toàn đen
    - số ảnh gần như toàn sáng
    """
    labels, images = image_arrays(df)
    pixels = images.astype(np.float32)

    ink_ratio = (pixels > 20).mean(axis=(1, 2))

    report = {
        "sample_rows": len(df),
        "pixel_min": int(pixels.min()),
        "pixel_max": int(pixels.max()),
        "pixel_mean": float(pixels.mean()),
        "pixel_std": float(pixels.std()),
        "low_ink_images (<1%)": int((ink_ratio < 0.01).sum()),
        "high_ink_images (>90%)": int((ink_ratio > 0.90).sum()),
    }

    return pd.Series(report)


def show_extreme_images(df, title, mode="low", n=8):
    labels, images = image_arrays(df)
    pixels = images.astype(np.float32)
    ink_ratio = (pixels > 20).mean(axis=(1, 2))

    if mode == "low":
        indices = np.argsort(ink_ratio)[:n]
    elif mode == "high":
        indices = np.argsort(ink_ratio)[-n:][::-1]
    else:
        raise ValueError("mode phải là 'low' hoặc 'high'.")

    print("Ink ratio:")
    for idx in indices:
        print(f"index={idx:5d} | label={labels[idx]} | ink_ratio={ink_ratio[idx]:.4f}")

    show_image_grid(
        df,
        f"{title} – các ảnh {mode} ink ratio",
        indices=indices
    )


# 3.3. Dataset 1 – EMNIST ByMerge

## 3.3.1. Mục tiêu phân tích

Các nội dung cần kiểm tra:

- kích thước và cấu trúc dữ liệu;
- số lớp;
- phân bố nhãn trên toàn bộ dataset;
- kích thước ảnh;
- giá trị pixel;
- ảnh mẫu;
- một số chỉ số chất lượng ảnh;
- bước chuẩn hóa và reshape cần thiết.

Do file lớn, visualization sử dụng một mẫu dữ liệu; phân bố nhãn có thể quét toàn bộ file theo chunk.

In [ ]:
emnist_sample, emnist_header = load_image_sample(EMNIST_PATH)

print("Có header:", emnist_header == 0)
print("Số dòng trong mẫu:", len(emnist_sample))
print("Số cột:", emnist_sample.shape[1])
print("Kích thước ảnh kỳ vọng:", "28 x 28")
print("Số pixel kỳ vọng:", emnist_sample.shape[1] - 1)

display(emnist_sample.head())
display(emnist_sample.dtypes.head(10))


In [ ]:
print("Đang quét toàn bộ cột nhãn của EMNIST ByMerge...")
emnist_label_counts = scan_label_distribution(EMNIST_PATH)

print("Tổng số mẫu:", int(emnist_label_counts.sum()))
print("Số lớp:", int(emnist_label_counts.size))

display(
    emnist_label_counts.rename("count").to_frame()
)


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
emnist_label_counts.plot(kind="bar", ax=ax)
ax.set_title("EMNIST ByMerge – phân bố nhãn trên toàn bộ dataset")
ax.set_xlabel("Label")
ax.set_ylabel("Số lượng mẫu")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

emnist_balance = pd.DataFrame({
    "count": emnist_label_counts,
    "percentage": 100 * emnist_label_counts / emnist_label_counts.sum()
}).sort_values("count")

display(emnist_balance.head(10))
display(emnist_balance.tail(10))


In [ ]:
show_image_grid(
    emnist_sample,
    "EMNIST ByMerge – ảnh mẫu"
)

display(
    image_quality_report(emnist_sample).to_frame("value")
)


### Kiểm tra các ảnh có đặc trưng bất thường

Ở đây **không xóa ảnh tự động**. Mục tiêu của cell này chỉ là tìm các mẫu có tỷ lệ pixel sáng quá thấp hoặc quá cao để xem xét bằng mắt.

Điều này giúp phân biệt giữa:

- ảnh hợp lệ nhưng nét chữ mảnh;
- ảnh trống/gần trống;
- ảnh có mức nền hoặc mực bất thường.

Quyết định loại bỏ ảnh thuộc bước xử lý dữ liệu sau khi đã quan sát.

In [ ]:
show_extreme_images(
    emnist_sample,
    "EMNIST ByMerge",
    mode="low",
    n=8
)


In [ ]:
show_extreme_images(
    emnist_sample,
    "EMNIST ByMerge",
    mode="high",
    n=8
)


### 3.3.2. Preprocessing cơ bản cho ảnh EMNIST

Ảnh CSV gồm:

`1 label + 784 pixel`

Do đó dữ liệu ảnh cần:

1. chuyển pixel sang `float32`;
2. chuẩn hóa từ `[0, 255]` về `[0, 1]`;
3. reshape từ 784 phần tử thành `28 × 28`;
4. thêm chiều channel khi cần dùng với CNN: `(N, 1, 28, 28)`.

Trong Phần 3 chỉ kiểm tra và tạo dạng dữ liệu sẵn sàng cho bước tiếp theo; **chưa huấn luyện mô hình**.

In [ ]:
def preprocess_image_sample(df):
    y = df["label"].to_numpy()
    X = df.drop(columns="label").to_numpy(dtype=np.float32)

    if X.shape[1] != 784:
        raise ValueError(f"Expected 784 pixel columns, got {X.shape[1]}.")

    X = X / 255.0
    X = X.reshape(-1, 1, 28, 28)

    return X, y

X_emnist_sample, y_emnist_sample = preprocess_image_sample(emnist_sample)

print("Shape sau reshape:", X_emnist_sample.shape)
print("Kiểu dữ liệu:", X_emnist_sample.dtype)
print("Min:", X_emnist_sample.min())
print("Max:", X_emnist_sample.max())


# 3.4. Dataset 2 – A–Z Handwritten Alphabets

## 3.4.1. Mục tiêu phân tích

Tập trung vào:

- cấu trúc CSV;
- số lớp A–Z;
- phân bố nhãn;
- ảnh mẫu;
- pixel;
- kiểm tra các mẫu có đặc trưng bất thường;
- chuẩn hóa và reshape.

Vì đây là dataset ảnh lớn, notebook vẫn dùng chiến lược **quét nhãn toàn bộ + sample để visualization**.

In [ ]:
az_sample, az_header = load_image_sample(AZ_PATH)

print("Có header:", az_header == 0)
print("Số dòng trong mẫu:", len(az_sample))
print("Số cột:", az_sample.shape[1])
print("Kích thước ảnh kỳ vọng:", "28 x 28")
print("Số pixel kỳ vọng:", az_sample.shape[1] - 1)

display(az_sample.head())
display(az_sample.dtypes.head(10))


In [ ]:
print("Đang quét toàn bộ cột nhãn của A-Z Handwritten...")
az_label_counts = scan_label_distribution(AZ_PATH)

print("Tổng số mẫu:", int(az_label_counts.sum()))
print("Số lớp:", int(az_label_counts.size))

display(
    az_label_counts.rename("count").to_frame()
)


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
az_label_counts.plot(kind="bar", ax=ax)
ax.set_title("A–Z Handwritten – phân bố nhãn trên toàn bộ dataset")
ax.set_xlabel("Label")
ax.set_ylabel("Số lượng mẫu")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

az_balance = pd.DataFrame({
    "count": az_label_counts,
    "percentage": 100 * az_label_counts / az_label_counts.sum()
}).sort_values("count")

display(az_balance)


In [ ]:
show_image_grid(
    az_sample,
    "A–Z Handwritten – ảnh mẫu"
)

display(
    image_quality_report(az_sample).to_frame("value")
)


### Kiểm tra các ảnh có khả năng bất thường

Dataset ảnh có thể xuất hiện các mẫu có quá ít hoặc quá nhiều pixel mực. Cell dưới đây chỉ **đánh dấu ứng viên để kiểm tra**, không tự động loại bỏ.

Đây là phần đặc biệt hữu ích cho preprocessing của dataset A–Z vì chất lượng ảnh có thể không hoàn toàn đồng nhất.

In [ ]:
show_extreme_images(
    az_sample,
    "A–Z Handwritten",
    mode="low",
    n=8
)


In [ ]:
show_extreme_images(
    az_sample,
    "A–Z Handwritten",
    mode="high",
    n=8
)


### 3.4.2. Preprocessing cơ bản

Quy trình giữ nhất quán với EMNIST:

`784 pixel → float32 → /255.0 → 28×28 → channel`

Chưa thực hiện data augmentation hoặc xây dựng mô hình ở phần này.

In [ ]:
X_az_sample, y_az_sample = preprocess_image_sample(az_sample)

print("Shape sau reshape:", X_az_sample.shape)
print("Kiểu dữ liệu:", X_az_sample.dtype)
print("Min:", X_az_sample.min())
print("Max:", X_az_sample.max())


# 3.5. Dataset 3 – Diabetes04

## 3.5.1. Tổng quan

Khác với hai dataset ảnh, Diabetes04 là dữ liệu dạng bảng.

Phần phân tích tập trung vào:

- kích thước dữ liệu;
- kiểu dữ liệu;
- thống kê mô tả;
- missing;
- duplicate;
- phân bố target;
- phân phối các biến số;
- correlation;
- kiểm tra outlier;
- tiền xử lý sau khi làm sạch.

**Chưa xây dựng mô hình dự đoán.**

In [ ]:
diabetes_df = pd.read_csv(DIABETES_PATH)

print("Shape:", diabetes_df.shape)
display(diabetes_df.head())

print("\nThông tin dataframe:")
print(diabetes_df.info())

display(diabetes_df.describe().T)


In [ ]:
TARGET_COL = "Diabetes_012"

if TARGET_COL not in diabetes_df.columns:
    raise KeyError(
        f"Không tìm thấy target '{TARGET_COL}'. "
        f"Các cột hiện có: {list(diabetes_df.columns)}"
    )

quality_diabetes = pd.DataFrame({
    "missing": diabetes_df.isna().sum(),
    "unique": diabetes_df.nunique(),
    "dtype": diabetes_df.dtypes.astype(str),
})

display(quality_diabetes)

print("Duplicate rows:", int(diabetes_df.duplicated().sum()))
print("Target:", TARGET_COL)
print("Số lớp target:", int(diabetes_df[TARGET_COL].nunique()))
display(diabetes_df[TARGET_COL].value_counts().sort_index().rename("count").to_frame())


In [ ]:
target_counts = diabetes_df[TARGET_COL].value_counts().sort_index()

target_percentage = 100 * target_counts / target_counts.sum()

target_summary = pd.DataFrame({
    "count": target_counts,
    "percentage": target_percentage.round(2)
})

display(target_summary)

fig, ax = plt.subplots(figsize=(7, 5))
target_counts.plot(kind="bar", ax=ax)
ax.set_title("Diabetes04 – phân bố target")
ax.set_xlabel("Diabetes_012")
ax.set_ylabel("Số lượng")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


### Nhận xét cần ghi nhận

Từ kiểm tra chất lượng, cần chú ý hai vấn đề:

1. **Duplicate**: các dòng trùng lặp cần được xác định và xử lý trước khi chuẩn bị dữ liệu.
2. **Class imbalance**: target có 3 lớp với quy mô không đồng đều, vì vậy đây là một đặc điểm quan trọng của dataset.

Missing value được kiểm tra riêng cho từng cột; nếu có missing thực tế thì phải ghi nhận trước khi quyết định cách xử lý.

In [ ]:
numeric_cols = diabetes_df.select_dtypes(include=np.number).columns.tolist()

# Không đưa target vào histogram đặc trưng.
feature_numeric_cols = [
    c for c in numeric_cols if c != TARGET_COL
]

# Hiển thị một nhóm biến để EDA không quá dày.
plot_cols = feature_numeric_cols[:12]

ncols = 3
nrows = int(np.ceil(len(plot_cols) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax in axes:
    ax.axis("off")

for ax, col in zip(axes, plot_cols):
    ax.hist(diabetes_df[col].dropna(), bins=30)
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.grid(axis="y", alpha=0.2)

plt.tight_layout()
plt.show()


### 3.5.2. Correlation giữa các biến số

Biểu đồ dưới đây giúp quan sát nhanh mối liên hệ tuyến tính giữa các thuộc tính số.

Correlation chỉ phục vụ **phân tích dữ liệu** ở phần này; không dùng nó để kết luận nguyên nhân–kết quả.

In [ ]:
corr = diabetes_df[feature_numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, aspect="auto")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ax.set_xticks(np.arange(len(corr.columns)))
ax.set_yticks(np.arange(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticklabels(corr.columns)

ax.set_title("Diabetes04 – ma trận tương quan")
plt.tight_layout()
plt.show()


### 3.5.3. Kiểm tra outlier bằng IQR

Outlier được **phát hiện và thống kê**, chưa tự động xóa.

Điều này quan trọng vì một giá trị cực đoan không nhất thiết là lỗi dữ liệu; cần phân biệt giữa:

- dữ liệu bất thường nhưng hợp lệ;
- dữ liệu nhập sai;
- giá trị thực sự hiếm.



In [ ]:
def iqr_outlier_summary(df, columns):
    rows = []

    for col in columns:
        s = df[col].dropna()

        q1 = s.quantile(0.25)
        q3 = s.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        mask = (s < lower) | (s > upper)

        rows.append({
            "feature": col,
            "Q1": q1,
            "Q3": q3,
            "IQR": iqr,
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_count": int(mask.sum()),
            "outlier_percentage": 100 * mask.mean(),
        })

    return pd.DataFrame(rows).sort_values(
        "outlier_percentage",
        ascending=False
    )

outlier_summary = iqr_outlier_summary(diabetes_df, feature_numeric_cols)

display(outlier_summary)


### 3.5.4. Làm sạch duplicate và chuẩn hóa dữ liệu

Bước làm sạch ở đây chỉ thực hiện những gì có bằng chứng rõ ràng từ kiểm tra:

- xóa các dòng trùng lặp hoàn toàn;
- giữ target riêng;
- giữ các biến số làm feature;
- chuẩn hóa feature bằng `StandardScaler`.

**Không tự ý xóa outlier trong cell này.** Outlier chỉ được báo cáo ở bước trước.

In [ ]:
diabetes_clean = diabetes_df.drop_duplicates().reset_index(drop=True)

print("Shape trước khi xóa duplicate :", diabetes_df.shape)
print("Shape sau khi xóa duplicate  :", diabetes_clean.shape)
print("Số dòng đã loại              :", len(diabetes_df) - len(diabetes_clean))

X_diabetes = diabetes_clean.drop(columns=[TARGET_COL])
y_diabetes = diabetes_clean[TARGET_COL].to_numpy()

numeric_features = X_diabetes.select_dtypes(include=np.number).columns.tolist()

scaler = StandardScaler()
X_diabetes_scaled = X_diabetes.copy()
X_diabetes_scaled[numeric_features] = scaler.fit_transform(
    X_diabetes[numeric_features]
)

print("\nShape X sau chuẩn bị:", X_diabetes_scaled.shape)
print("Shape y:", y_diabetes.shape)

display(X_diabetes_scaled.head())


# 3.6. Tổng hợp đặc điểm 3 dataset

Phần này chỉ tổng hợp những gì đã quan sát được trong notebook.

| Dataset | Loại dữ liệu | Đơn vị ảnh / feature | Target | Vấn đề cần chú ý | Preprocessing chính |
|---|---|---|---|---|---|
| EMNIST ByMerge | Ảnh grayscale | 28×28 = 784 pixel | Label ký tự | Quy mô rất lớn, phân bố lớp cần kiểm tra, kiểm tra ảnh bất thường | Normalize pixel, reshape, kiểm tra ảnh bất thường |
| A–Z Handwritten | Ảnh grayscale | 28×28 = 784 pixel | Label A–Z | Kiểm tra chất lượng ảnh và phân bố lớp | Normalize pixel, reshape, kiểm tra ảnh bất thường |
| Diabetes04 | Tabular | Các thuộc tính số | `Diabetes_012` | Duplicate, mất cân bằng lớp, outlier | Xóa duplicate, kiểm tra outlier, StandardScaler |

> Lưu ý: kết quả số cụ thể của 3 dataset được sinh ra trực tiếp khi notebook chạy trên các file trong `C:\DATA\`.

In [ ]:
summary = pd.DataFrame([
    {
        "Dataset": "EMNIST ByMerge",
        "Type": "Ảnh grayscale",
        "Input": "28 x 28",
        "Target": "Label ký tự",
        "Quality focus": "Class distribution + image quality",
        "Preprocessing": "Normalize + reshape"
    },
    {
        "Dataset": "A-Z Handwritten",
        "Type": "Ảnh grayscale",
        "Input": "28 x 28",
        "Target": "A-Z label",
        "Quality focus": "Class distribution + image quality",
        "Preprocessing": "Normalize + reshape"
    },
    {
        "Dataset": "Diabetes04",
        "Type": "Tabular",
        "Input": f"{len(diabetes_clean.columns)-1} features sau khi tách target",
        "Target": TARGET_COL,
        "Quality focus": "Duplicate + imbalance + outlier",
        "Preprocessing": "Deduplicate + StandardScaler"
    },
])

display(summary)


# 3.7. Kết luận Phần 3

Sau khi hoàn thành notebook, cần có đủ các nội dung:

- xác định rõ cấu trúc của từng dataset;
- thống kê kích thước và phân bố target;
- kiểm tra dữ liệu thiếu và dữ liệu trùng;
- phân tích chất lượng dữ liệu;
- visualization dữ liệu;
- xác định các bước preprocessing cần thiết;
- tạo dữ liệu ở dạng phù hợp để chuyển sang bước xây dựng mô hình.

**Phần này dừng tại đây.** Không xây dựng CNN, không huấn luyện mô hình và không thực hiện đánh giá mô hình trong notebook này.